# S8 - Actividad autónoma: streaming con Spark sobre `citibike-eventos`

**Proyecto Sello:** CitiBike Intelligent Mobility · **Fuente real:** el topic `citibike-eventos` construido en S7 (módulo `uso-citibike/`), que recibe dos eventos reales:

- `bici.vibracion`: acelerómetro del celular (phyphox) → MQTT → Kafka, 1 lectura por segundo, key `bici-grimaldo-01`.
- `estacion.estado`: feed GBFS real de Citi Bike, 6 estaciones cada 30 s, key = `stationId`.

Cada sección corresponde a una tarea de 4.1. Antes de ejecutar, los cuatro scripts de `uso-citibike` deben estar corriendo (desde la raíz de `lambda26`, en PowerShell):

```powershell
docker compose -f kafka/compose.yml up -d
docker compose -f uso-citibike/compose.yml up -d
docker compose -f pyspark/compose.yml -f pyspark/compose.kafka.yml up -d
docker exec -d lambda26-uso-citibike sh -c "python -u /app/consumer_citibike.py > /app/logs/consumer.log 2>&1"
docker exec -d lambda26-uso-citibike sh -c "python -u /app/bridge_mqtt_kafka.py > /app/logs/bridge.log 2>&1"
docker exec -d lambda26-uso-citibike sh -c "python -u /app/lector_phyphox_mqtt.py > /app/logs/lector.log 2>&1"
docker exec -d lambda26-uso-citibike sh -c "python -u /app/producer_gbfs.py > /app/logs/gbfs.log 2>&1"
```

## 0. `SparkSession` con el conector de Kafka y lectura batch del topic

Misma `SparkSession` de la guía 3.2: conector `spark-sql-kafka-0-10_2.13:4.2.0`, `shuffle.partitions = 3` (las 3 particiones de `citibike-eventos`) y Spark UI en el puerto 4040 del contenedor (en la laptop se abre en `http://localhost:4042`, por el mapeo `4042:4040` de `compose.yml`).

Única línea agregada: `spark.sql.session.timeZone = America/Lima`. El contenedor corre en UTC; sin esta línea, las ventanas se mostrarían 5 horas adelantadas respecto del reloj de la laptop que aparece en las capturas.

In [1]:
import os
import json
import time
import shutil
import threading
from datetime import datetime, timezone, timedelta

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion8-streaming-citibike-autonoma")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.ui.port", "4040")
    .config("spark.sql.session.timeZone", "America/Lima")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

BOOTSTRAP = "kafka:9092"
TOPIC = "citibike-eventos"
BIKE_ID = "bici-grimaldo-01"
LIMA = timezone(timedelta(hours=-5))

resumen = {}  # resultados clave de cada sección, para la tabla final

def fecha_legible(ms):
    return datetime.fromtimestamp(ms / 1000, LIMA).strftime("%Y-%m-%d %H:%M:%S.%f")[:-3] + " (Lima)"

def limpiar(ruta):
    # Cada consulta de este notebook arranca con su checkpoint vacío: si se reutilizara uno
    # viejo, Spark retomaría desde sus offsets y no desde 'latest' (ver sección 5).
    shutil.rmtree(ruta, ignore_errors=True)

spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-44562ce5-6d29-4e66-8e0c-10c43264ebdc;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central


	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 441ms :: artifacts dl 16ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.slf4j#slf4j-api;2.0

26/10/09 04:10:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Lectura **batch** de todo lo que hoy contiene `citibike-eventos`: total de mensajes, y por cada una de las 3 particiones cuántos mensajes y qué keys tiene. Las keys deben ser `bici-grimaldo-01` (vibración) y los `stationId` del feed GBFS: la key decide la partición, así que todas las lecturas de la bici caen siempre en la misma.

In [2]:
from pyspark.sql.functions import col, count, countDistinct, collect_set, get_json_object

def leer_topic_batch():
    return (
        spark.read.format("kafka")
        .option("kafka.bootstrap.servers", BOOTSTRAP)
        .option("subscribe", TOPIC)
        .option("startingOffsets", "earliest")
        .load()
    )

lote = leer_topic_batch()
total_mensajes = lote.count()
print("mensajes en el topic:", total_mensajes)

por_particion = (
    lote.selectExpr("partition", "offset", "CAST(key AS STRING) AS key")
    .groupBy("partition")
    .agg(count("*").alias("mensajes"),
         countDistinct("key").alias("keys_distintas"),
         collect_set("key").alias("keys"))
    .orderBy("partition")
)
por_particion.show(truncate=False)

lote.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value", "partition", "offset") \
    .orderBy("offset", ascending=False).show(3, truncate=90)

resumen["0. Lectura batch"] = f"{total_mensajes} mensajes en {por_particion.count()} particiones"

mensajes en el topic: 1137


+---------+--------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|partition|mensajes|keys_distintas|keys                                                                                                                                                            |
+---------+--------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|0        |74      |6             |[66de2274-0aca-11e7-82f6-3863bb44ef7c, 1986707977001495180, 2235290248874043146, 2235294316208072496, 06439006-11b6-44f0-8545-c9d39035f32a, 1869753484503987044]|
|1        |1028    |4             |[1822663031356509142, bici-grimaldo-01, 525535e3-dde3-47b4-9c2e-0174eaa4f2bb, 2235294222661977076]                                                              |
|2        |37  

+----------------+------------------------------------------------------------------------------------------+---------+------+
|             key|                                                                                     value|partition|offset|
+----------------+------------------------------------------------------------------------------------------+---------+------+
|bici-grimaldo-01|{"tipoEvento": "bici.vibracion", "bikeId": "bici-grimaldo-01", "accX": -0.015, "accY": ...|        1|  2732|
|bici-grimaldo-01|{"tipoEvento": "bici.vibracion", "bikeId": "bici-grimaldo-01", "accX": 0.01, "accY": -0...|        1|  2731|
|bici-grimaldo-01|{"tipoEvento": "bici.vibracion", "bikeId": "bici-grimaldo-01", "accX": -0.006, "accY": ...|        1|  2730|
+----------------+------------------------------------------------------------------------------------------+---------+------+
only showing top 3 rows


## 1. Lectura streaming con esquema explícito

`readStream` con `startingOffsets = "latest"`: solo interesa lo que llega desde ahora. El esquema es **explícito** y reúne los campos de los **dos** eventos del contrato (`uso-citibike/CONTRATO.md`): en cada fila, los campos del otro tipo quedan en `null`. `ts` convierte `timestamp` (milisegundos epoch) a un `timestamp` real: es el **tiempo de evento** que usan todas las ventanas de este notebook.

In [3]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import StructType, StringType, DoubleType, LongType, IntegerType

crudo = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "latest")
    .load()
)
print("es un stream:", crudo.isStreaming)

esquema_evento = (
    StructType()
    .add("tipoEvento", StringType())
    # bici.vibracion (acelerómetro del celular, phyphox)
    .add("bikeId", StringType())
    .add("accX", DoubleType())
    .add("accY", DoubleType())
    .add("accZ", DoubleType())
    .add("magnitud", DoubleType())
    .add("unidad", StringType())
    # comunes
    .add("origen", StringType())
    .add("timestamp", LongType())
    # estacion.estado (feed GBFS de Citi Bike)
    .add("stationId", StringType())
    .add("bicisDisponibles", IntegerType())
    .add("docksDisponibles", IntegerType())
)

eventos = (
    crudo.selectExpr("CAST(value AS STRING) AS json_str")
    .select(from_json(col("json_str"), esquema_evento).alias("e"))
    .select("e.*")
    .withColumn("ts", (col("timestamp") / 1000).cast("timestamp"))
)
eventos.printSchema()

vibracion = eventos.filter(col("tipoEvento") == "bici.vibracion")
estaciones = eventos.filter(col("tipoEvento") == "estacion.estado")
print("vibracion es stream:", vibracion.isStreaming, "| estaciones es stream:", estaciones.isStreaming)

resumen["1. Esquema"] = f"{len(esquema_evento.fields)} campos + ts; 2 DataFrames (vibracion, estaciones)"

es un stream: True
root
 |-- tipoEvento: string (nullable = true)
 |-- bikeId: string (nullable = true)
 |-- accX: double (nullable = true)
 |-- accY: double (nullable = true)
 |-- accZ: double (nullable = true)
 |-- magnitud: double (nullable = true)
 |-- unidad: string (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- stationId: string (nullable = true)
 |-- bicisDisponibles: integer (nullable = true)
 |-- docksDisponibles: integer (nullable = true)
 |-- ts: timestamp (nullable = true)

vibracion es stream: True | estaciones es stream: True


## 2. Agregación por ventana con watermark

### 2.1 Vibración de la bici: ventana fija de 10 s con watermark de 20 s

Métrica del dominio: cuánto vibra la bici en cada bloque de 10 s (promedio y pico de `magnitud`, y cuántas lecturas entraron). Un pico alto sostenido indica pavimento en mal estado o un golpe.

**Por qué 20 segundos de watermark**, con datos reales de S7:

- El celular publica **1 lectura por segundo**, así que una ventana de 10 s junta unas 10 lecturas.
- La latencia medida punta a punta (celular → MQTT → Kafka → consumer) fue de **200-430 ms** en S7 (hoy, al arrancar S8, ~230 ms). Eso es 50 veces menos que el margen.
- El broker público `test.mosquitto.org` puede atrasarse varios segundos (en S7 tardó en conectar). 20 s absorbe esos atrasos sin perder lecturas.
- A la vez, 20 s mantiene abiertas solo unas 3 ventanas de 10 s por bici (la actual y las 2 anteriores). No se guarda estado de más, y el resultado de cada ventana se da por cerrado a los ~30 s de su inicio.

In [4]:
from pyspark.sql.functions import window, avg, max as max_

CHK_VENTANAS = "/tmp/s08-citibike-chk-ventanas"
limpiar(CHK_VENTANAS)

vibracion_ventanas = (
    vibracion
    .withWatermark("ts", "20 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("bikeId"))
    .agg(avg("magnitud").alias("mag_prom"),
         max_("magnitud").alias("mag_max"),
         count("*").alias("n"))
)

consulta = (
    vibracion_ventanas.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("checkpointLocation", CHK_VENTANAS)
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(40)
progreso = consulta.lastProgress
consulta.stop()

print("último batchId:", progreso["batchId"])
print("watermark al final (UTC):", progreso["eventTime"].get("watermark"))
print("evento más reciente visto (UTC):", progreso["eventTime"].get("max"))
resumen["2. Ventana 10s + watermark 20s"] = f"{progreso['batchId'] + 1} micro-lotes en 40 s, outputMode update"

-------------------------------------------
Batch: 0
-------------------------------------------
+------+------+--------+-------+---+
|window|bikeId|mag_prom|mag_max|n  |
+------+------+--------+-------+---+
+------+------+--------+-------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------------+--------+-------+---+
|window                                    |bikeId          |mag_prom|mag_max|n  |
+------------------------------------------+----------------+--------+-------+---+
|{2026-10-08 23:10:10, 2026-10-08 23:10:20}|bici-grimaldo-01|0.0445  |0.08   |4  |
+------------------------------------------+----------------+--------+-------+---+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------------+--------+-------+---+
|window                                    |bikeId          |mag_prom|mag_max|n  |
+------------------------------------------+----------------+--------+-------+---+
|{2026-10-08 23:10:10, 2026-10-08 23:10:20}|bici-grimaldo-01|0.0434  |0.08   |5  |
|{2026-10-08 23:10:20, 2026-10-08 23:10:30}|bici-grimaldo-01|0.04175 |0.076  |4  |
+------------------------------------------+----------------+--------+-------+---+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------------+--------------------+-------+---+
|window                                    |bikeId          |mag_prom            |mag_max|n  |
+------------------------------------------+----------------+--------------------+-------+---+
|{2026-10-08 23:10:20, 2026-10-08 23:10:30}|bici-grimaldo-01|0.037125000000000005|0.076  |8  |
+------------------------------------------+----------------+--------------------+-------+---+



-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+----------------+-------------------+-------+---+
|window                                    |bikeId          |mag_prom           |mag_max|n  |
+------------------------------------------+----------------+-------------------+-------+---+
|{2026-10-08 23:10:30, 2026-10-08 23:10:40}|bici-grimaldo-01|0.0325             |0.057  |4  |
|{2026-10-08 23:10:20, 2026-10-08 23:10:30}|bici-grimaldo-01|0.03911111111111112|0.076  |9  |
+------------------------------------------+----------------+-------------------+-------+---+



-------------------------------------------
Batch: 5
-------------------------------------------
+------------------------------------------+----------------+-------------------+-------+---+
|window                                    |bikeId          |mag_prom           |mag_max|n  |
+------------------------------------------+----------------+-------------------+-------+---+
|{2026-10-08 23:10:30, 2026-10-08 23:10:40}|bici-grimaldo-01|0.03911111111111111|0.111  |9  |
+------------------------------------------+----------------+-------------------+-------+---+



-------------------------------------------
Batch: 6
-------------------------------------------
+------------------------------------------+----------------+--------+-------+---+
|window                                    |bikeId          |mag_prom|mag_max|n  |
+------------------------------------------+----------------+--------+-------+---+
|{2026-10-08 23:10:40, 2026-10-08 23:10:50}|bici-grimaldo-01|0.0855  |0.2    |4  |
+------------------------------------------+----------------+--------+-------+---+



-------------------------------------------
Batch: 7
-------------------------------------------
+------------------------------------------+----------------+--------------------+-------+---+
|window                                    |bikeId          |mag_prom            |mag_max|n  |
+------------------------------------------+----------------+--------------------+-------+---+
|{2026-10-08 23:10:40, 2026-10-08 23:10:50}|bici-grimaldo-01|0.059777777777777784|0.2    |9  |
+------------------------------------------+----------------+--------------------+-------+---+



último batchId: 7
watermark al final (UTC): 2026-10-09T04:10:24.018Z
evento más reciente visto (UTC): 2026-10-09T04:10:49.386Z


Con `outputMode("update")`, cada micro-lote imprime **solo las ventanas que cambiaron**. Una ventana se repite mientras recibe lecturas (con `n` creciendo hasta ~10). Cuando el watermark pasa su final, deja de aparecer y su estado se libera de la memoria.

### 2.2 Utilización de estaciones: ventana deslizante de 2 min con paso de 30 s

Indicador de utilización: promedio de `bicisDisponibles` por estación en ventanas de 2 minutos que se recalculan cada 30 s (el ritmo de publicación de GBFS). `ocupacion_pct` = bicis / (bicis + anclajes libres). Una estación cerca de 0 % está vacía (no hay bicis para retirar) y una cerca de 100 % está llena (no hay dónde devolverlas).

Watermark de **1 minuto**: GBFS publica cada 30 s, así que un margen de dos ciclos tolera que una publicación se atrase una vuelta completa sin perderla. La consulta corre 65 s para alcanzar al menos 2 publicaciones de GBFS.

In [5]:
from pyspark.sql.functions import expr

CHK_ESTACIONES = "/tmp/s08-citibike-chk-estaciones"
limpiar(CHK_ESTACIONES)

estaciones_deslizante = (
    estaciones
    .withWatermark("ts", "1 minute")
    .groupBy(window(col("ts"), "2 minutes", "30 seconds"), col("stationId"))
    .agg(avg("bicisDisponibles").alias("bicis_prom"),
         avg("docksDisponibles").alias("docks_prom"),
         count("*").alias("n"))
    .withColumn("ocupacion_pct", expr("round(100 * try_divide(bicis_prom, bicis_prom + docks_prom), 1)"))
)

consulta = (
    estaciones_deslizante.writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .option("numRows", 60)
    .option("checkpointLocation", CHK_ESTACIONES)
    .trigger(processingTime="5 seconds")
    .start()
)
consulta.awaitTermination(65)
progreso_est = consulta.lastProgress
consulta.stop()
print("último batchId:", progreso_est["batchId"], "| watermark (UTC):", progreso_est["eventTime"].get("watermark"))
resumen["2. Deslizante estaciones 2min/30s"] = f"{progreso_est['batchId'] + 1} micro-lotes en 65 s, watermark 1 min"

-------------------------------------------
Batch: 0
-------------------------------------------
+------+---------+----------+----------+---+-------------+
|window|stationId|bicis_prom|docks_prom|n  |ocupacion_pct|
+------+---------+----------+----------+---+-------------+
+------+---------+----------+----------+---+-------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+------------------------------------+----------+----------+---+-------------+
|window                                    |stationId                           |bicis_prom|docks_prom|n  |ocupacion_pct|
+------------------------------------------+------------------------------------+----------+----------+---+-------------+
|{2026-10-08 23:09:30, 2026-10-08 23:11:30}|2235294316208072496                 |11.0      |5.0       |1  |68.8         |
|{2026-10-08 23:10:30, 2026-10-08 23:12:30}|66de2274-0aca-11e7-82f6-3863bb44ef7c|23.0      |0.0       |1  |100.0        |
|{2026-10-08 23:09:30, 2026-10-08 23:11:30}|66de2274-0aca-11e7-82f6-3863bb44ef7c|23.0      |0.0       |1  |100.0        |
|{2026-10-08 23:09:30, 2026-10-08 23:11:30}|1869753484503987044                 |4.0       |11.0      |1  |26.7         |
|{2026-10-08 23:09:00, 2026-10-08 23:11:00}|21244

-------------------------------------------
Batch: 2
-------------------------------------------
+------+---------+----------+----------+---+-------------+
|window|stationId|bicis_prom|docks_prom|n  |ocupacion_pct|
+------+---------+----------+----------+---+-------------+
+------+---------+----------+----------+---+-------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+------+---------+----------+----------+---+-------------+
|window|stationId|bicis_prom|docks_prom|n  |ocupacion_pct|
+------+---------+----------+----------+---+-------------+
+------+---------+----------+----------+---+-------------+



-------------------------------------------
Batch: 4
-------------------------------------------
+------+---------+----------+----------+---+-------------+
|window|stationId|bicis_prom|docks_prom|n  |ocupacion_pct|
+------+---------+----------+----------+---+-------------+
+------+---------+----------+----------+---+-------------+



...

-------------------------------------------
Batch: 8
-------------------------------------------
+------+---------+----------+----------+---+-------------+
|window|stationId|bicis_prom|docks_prom|n  |ocupacion_pct|
+------+---------+----------+----------+---+-------------+
+------+---------+----------+----------+---+-------------+



-------------------------------------------
Batch: 9
-------------------------------------------
+------+---------+----------+----------+---+-------------+
|window|stationId|bicis_prom|docks_prom|n  |ocupacion_pct|
+------+---------+----------+----------+---+-------------+
+------+---------+----------+----------+---+-------------+



-------------------------------------------
Batch: 10
-------------------------------------------
+------+---------+----------+----------+---+-------------+
|window|stationId|bicis_prom|docks_prom|n  |ocupacion_pct|
+------+---------+----------+----------+---+-------------+
+------+---------+----------+----------+---+-------------+



-------------------------------------------
Batch: 11
-------------------------------------------
+------+---------+----------+----------+---+-------------+
|window|stationId|bicis_prom|docks_prom|n  |ocupacion_pct|
+------+---------+----------+----------+---+-------------+
+------+---------+----------+----------+---+-------------+



último batchId: 11 | watermark (UTC): 2026-10-09T04:10:27.625Z


Cada lectura de una estación aparece en **4 ventanas** de 2 minutos a la vez (2 min / 30 s = 4 ventanas solapadas), por eso el mismo `stationId` se repite con rangos de `window` que se superponen.

## 3. Dato tardío real, descartado por el watermark

Para no caer en el **arranque en frío** (guía 3.10), primero se deja correr la consulta 25 s con datos en vivo, y recién después el hilo reenvía el evento tardío. Así el watermark ya avanzó.

El evento tardío es **real y sin modificar**: el hilo busca con lectura batch la lectura **más antigua** de `bici-grimaldo-01` que todavía conserva el topic y reenvía al mismo topic sus bytes exactos (`key` y `value`). El plan original era usar una lectura de la corrida de S7 (1 de octubre de 2026). Pero Kafka tiene `retention.ms = 604800000` (7 días) y esos segmentos ya se borraron: ver `uso-citibike/S08_hallazgos.md`.

Se usan dos sinks en paralelo sobre la misma agregación: `console` (para la captura) y `memory` (tabla `tardio_mem`, para comprobar con SQL).

In [6]:
CHK_TARDIO = "/tmp/s08-citibike-chk-tardio"
CHK_TARDIO_MEM = "/tmp/s08-citibike-chk-tardio-mem"
limpiar(CHK_TARDIO)
limpiar(CHK_TARDIO_MEM)

evento_tardio = {}

def lecturas_de_la_bici():
    return (
        leer_topic_batch()
        .filter(col("key").cast("string") == BIKE_ID)
        .withColumn("t", get_json_object(col("value").cast("string"), "$.timestamp").cast("long"))
    )

def publicar_bytes(key_bytes, value_bytes, veces=1):
    # Reenvía key y value EXACTOS (binarios, tal como estaban en Kafka): no se modifica nada.
    df = spark.createDataFrame([(key_bytes, value_bytes)], "key binary, value binary")
    for _ in range(veces):
        (df.write.format("kafka")
            .option("kafka.bootstrap.servers", BOOTSTRAP)
            .option("topic", TOPIC)
            .save())

def publicar_evento_tardio(segundos_de_espera):
    time.sleep(segundos_de_espera)
    fila = lecturas_de_la_bici().orderBy("t", "offset").first()
    publicar_bytes(fila["key"], fila["value"])
    ahora_ms = int(time.time() * 1000)
    evento_tardio.update(timestamp=fila["t"], value=bytes(fila["value"]).decode(),
                         particion=fila["partition"], offset=fila["offset"], publicado_ms=ahora_ms)
    print(f">>> publicado evento tardío real | timestamp={fila['t']} | {fecha_legible(fila['t'])} "
          f"| {(ahora_ms - fila['t']) / 60000:.1f} min de atraso | original en partición {fila['partition']}, offset {fila['offset']}")
    print(">>> value reenviado sin modificar:", evento_tardio["value"])

agregado_tardio = (
    vibracion
    .withWatermark("ts", "20 seconds")
    .groupBy(window(col("ts"), "10 seconds"), col("bikeId"))
    .agg(count("*").alias("n"), avg("magnitud").alias("mag_prom"))
)

inicio_ms = int(time.time() * 1000)
hilo = threading.Thread(target=publicar_evento_tardio, args=(25,), daemon=True)
hilo.start()

q_consola = (
    agregado_tardio.writeStream
    .outputMode("update").format("console").option("truncate", "false")
    .option("checkpointLocation", CHK_TARDIO)
    .trigger(processingTime="4 seconds")
    .start()
)
q_memoria = (
    agregado_tardio.writeStream
    .outputMode("update").format("memory").queryName("tardio_mem")
    .option("checkpointLocation", CHK_TARDIO_MEM)
    .trigger(processingTime="4 seconds")
    .start()
)
q_consola.awaitTermination(65)
hilo.join()
progreso_tardio = q_consola.lastProgress
lotes_despues = [p["batchId"] for p in q_consola.recentProgress
                 if p["batchId"] > 0 and datetime.fromisoformat(p["timestamp"].replace("Z", "+00:00")).timestamp() * 1000 > evento_tardio["publicado_ms"]]
q_consola.stop()
q_memoria.stop()
print(f"micro-lotes procesados después de publicar el tardío: {len(lotes_despues)}")

-------------------------------------------
Batch: 0
-------------------------------------------
+------+------+---+--------+
|window|bikeId|n  |mag_prom|
+------+------+---+--------+
+------+------+---+--------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+----------------+---+--------+
|window                                    |bikeId          |n  |mag_prom|
+------------------------------------------+----------------+---+--------+
|{2026-10-08 23:11:50, 2026-10-08 23:12:00}|bici-grimaldo-01|2  |0.047   |
+------------------------------------------+----------------+---+--------+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+----------------+---+--------------------+
|window                                    |bikeId          |n  |mag_prom            |
+------------------------------------------+----------------+---+--------------------+
|{2026-10-08 23:12:00, 2026-10-08 23:12:10}|bici-grimaldo-01|4  |0.026250000000000002|
+------------------------------------------+----------------+---+--------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+----------------+---+-------------------+
|window                                    |bikeId          |n  |mag_prom           |
+------------------------------------------+----------------+---+-------------------+
|{2026-10-08 23:12:00, 2026-10-08 23:12:10}|bici-grimaldo-01|7  |0.04157142857142858|
+------------------------------------------+----------------+---+-------------------+



-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+----------------+---+--------------------+
|window                                    |bikeId          |n  |mag_prom            |
+------------------------------------------+----------------+---+--------------------+
|{2026-10-08 23:12:10, 2026-10-08 23:12:20}|bici-grimaldo-01|2  |0.0225              |
|{2026-10-08 23:12:00, 2026-10-08 23:12:10}|bici-grimaldo-01|9  |0.052444444444444446|
+------------------------------------------+----------------+---+--------------------+



...

-------------------------------------------
Batch: 13
-------------------------------------------
+------------------------------------------+----------------+---+-------------------+
|window                                    |bikeId          |n  |mag_prom           |
+------------------------------------------+----------------+---+-------------------+
|{2026-10-08 23:12:40, 2026-10-08 23:12:50}|bici-grimaldo-01|7  |0.04257142857142857|
+------------------------------------------+----------------+---+-------------------+



-------------------------------------------
Batch: 14
-------------------------------------------
+------------------------------------------+----------------+---+--------------------+
|window                                    |bikeId          |n  |mag_prom            |
+------------------------------------------+----------------+---+--------------------+
|{2026-10-08 23:12:40, 2026-10-08 23:12:50}|bici-grimaldo-01|9  |0.042777777777777776|
|{2026-10-08 23:12:50, 2026-10-08 23:13:00}|bici-grimaldo-01|2  |0.0415              |
+------------------------------------------+----------------+---+--------------------+



-------------------------------------------
Batch: 15
-------------------------------------------
+------------------------------------------+----------------+---+--------+
|window                                    |bikeId          |n  |mag_prom|
+------------------------------------------+----------------+---+--------+
|{2026-10-08 23:12:50, 2026-10-08 23:13:00}|bici-grimaldo-01|5  |0.0334  |
+------------------------------------------+----------------+---+--------+



-------------------------------------------
Batch: 16
-------------------------------------------
+------------------------------------------+----------------+---+-------------------+
|window                                    |bikeId          |n  |mag_prom           |
+------------------------------------------+----------------+---+-------------------+
|{2026-10-08 23:12:50, 2026-10-08 23:13:00}|bici-grimaldo-01|9  |0.03688888888888889|
+------------------------------------------+----------------+---+-------------------+



micro-lotes procesados después de publicar el tardío: 9


Comprobación programática: ¿apareció en la salida la ventana a la que pertenece el evento tardío, o alguna ventana anterior al inicio de esta consulta? La tabla `tardio_mem` guarda todo lo que el sink `memory` emitió, en todos los micro-lotes. También se confirma con lectura batch que el evento sí está en Kafka: que falte en la salida no se debe a que nunca se publicó.

In [7]:
t_ms = evento_tardio["timestamp"]

ventana_del_tardio = spark.sql(f'''
    SELECT * FROM tardio_mem
    WHERE window.start <= timestamp_millis({t_ms}) AND window.end > timestamp_millis({t_ms})
''')
ventanas_viejas = spark.sql(f'''
    SELECT * FROM tardio_mem WHERE window.end <= timestamp_millis({inicio_ms - 60000})
''')
rango = spark.sql("SELECT min(window.start) AS primera_ventana, max(window.end) AS ultima_ventana, count(*) AS filas_emitidas FROM tardio_mem")

copias_en_kafka = lecturas_de_la_bici().filter(col("t") == t_ms).count()

print("evento tardío:", t_ms, "|", fecha_legible(t_ms))
print("watermark al final de la consulta (UTC):", progreso_tardio["eventTime"].get("watermark"))
print("copias de ese evento en Kafka (original + un reenvío por cada ejecución de esta celda):", copias_en_kafka)
print("filas en la salida para la ventana del evento tardío:", ventana_del_tardio.count())
print("filas en la salida con ventanas anteriores al inicio de la consulta:", ventanas_viejas.count())
rango.show(truncate=False)

descartado = ventana_del_tardio.count() == 0 and ventanas_viejas.count() == 0
print("RESULTADO:", "el watermark DESCARTÓ el evento tardío en silencio" if descartado else "el evento tardío APARECIÓ en la salida")
resumen["3. Dato tardío"] = f"ts {fecha_legible(t_ms)}: {'descartado (0 filas)' if descartado else 'NO descartado'}"

evento tardío: 1791517959106 | 2026-10-08 22:52:39.106 (Lima)
watermark al final de la consulta (UTC): 2026-10-09T04:12:34.854Z
copias de ese evento en Kafka (original + un reenvío por cada ejecución de esta celda): 3
filas en la salida para la ventana del evento tardío: 0
filas en la salida con ventanas anteriores al inicio de la consulta: 0


+-------------------+-------------------+--------------+
|primera_ventana    |ultima_ventana     |filas_emitidas|
+-------------------+-------------------+--------------+
|2026-10-08 23:11:50|2026-10-08 23:13:00|19            |
+-------------------+-------------------+--------------+

RESULTADO: el watermark DESCARTÓ el evento tardío en silencio


## 4. Duplicado real, filtrado por `dropDuplicates` acotado por watermark

La consulta deduplica por (`bikeId`, `timestamp`) con watermark de 30 s, en `outputMode("append")`, con sink `console` y sink `memory` (`dedup_mem`). Mientras corre, se toma con lectura batch la lectura **real más reciente** de la bici y se reenvía **dos veces más**, con los mismos bytes. En Kafka quedan 3 copias (la original y 2 reenvíos), que es lo que pasa con un productor que reintenta: semántica *at-least-once*.

In [8]:
CHK_DEDUP = "/tmp/s08-citibike-chk-dedup"
CHK_DEDUP_MEM = "/tmp/s08-citibike-chk-dedup-mem"
limpiar(CHK_DEDUP)
limpiar(CHK_DEDUP_MEM)

deduplicado = (
    vibracion
    .withWatermark("ts", "30 seconds")
    .dropDuplicates(["bikeId", "timestamp"])
)

q_dedup = (
    deduplicado.writeStream
    .outputMode("append").format("console").option("truncate", "false")
    .option("checkpointLocation", CHK_DEDUP)
    .trigger(processingTime="3 seconds")
    .start()
)
q_dedup_mem = (
    deduplicado.writeStream
    .outputMode("append").format("memory").queryName("dedup_mem")
    .option("checkpointLocation", CHK_DEDUP_MEM)
    .trigger(processingTime="3 seconds")
    .start()
)

time.sleep(12)  # que ambas consultas ya estén leyendo en vivo
fila = lecturas_de_la_bici().orderBy(col("t").desc(), col("offset").desc()).first()
t_dup = fila["t"]
publicar_bytes(fila["key"], fila["value"], veces=2)
print(f">>> lectura real {t_dup} ({fecha_legible(t_dup)}) reenviada DOS veces más, sin modificar")
print(">>> value:", bytes(fila["value"]).decode())

q_dedup.awaitTermination(20)
q_dedup.stop()
q_dedup_mem.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+----------+------+----+----+----+--------+------+------+---------+---------+----------------+----------------+---+
|tipoEvento|bikeId|accX|accY|accZ|magnitud|unidad|origen|timestamp|stationId|bicisDisponibles|docksDisponibles|ts |
+----------+------+----+----+----+--------+------+------+---------+---------+----------------+----------------+---+
+----------+------+----+----+----+--------+------+------+---------+---------+----------------+----------------+---+



-------------------------------------------
Batch: 1
-------------------------------------------
+--------------+----------------+-----+------+-----+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|tipoEvento    |bikeId          |accX |accY  |accZ |magnitud|unidad|origen |timestamp    |stationId|bicisDisponibles|docksDisponibles|ts                     |
+--------------+----------------+-----+------+-----+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|0.014|-0.026|0.025|0.039   |m/s2  |phyphox|1791519182281|NULL     |NULL            |NULL            |2026-10-08 23:13:02.281|
+--------------+----------------+-----+------+-----+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|tipoEvento    |bikeId          |accX  |accY  |accZ  |magnitud|unidad|origen |timestamp    |stationId|bicisDisponibles|docksDisponibles|ts                     |
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|-0.021|0.032 |-0.035|0.052   |m/s2  |phyphox|1791519183434|NULL     |NULL            |NULL            |2026-10-08 23:13:03.434|
|bici.vibracion|bici-grimaldo-01|0.041 |-0.101|0.097 |0.145   |m/s2  |phyphox|1791519184594|NULL     |NULL            |NULL            |2026-10-08 23:13:04.594|
|bici.vibracion|bici-grimaldo-01|-0.009|-0.028|0.02  |0.036   |m/s2  |phyphox|1791519185747|NULL  

-------------------------------------------
Batch: 3
-------------------------------------------
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|tipoEvento    |bikeId          |accX  |accY  |accZ  |magnitud|unidad|origen |timestamp    |stationId|bicisDisponibles|docksDisponibles|ts                     |
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|-0.016|-0.006|0.002 |0.017   |m/s2  |phyphox|1791519186799|NULL     |NULL            |NULL            |2026-10-08 23:13:06.799|
|bici.vibracion|bici-grimaldo-01|-0.004|-0.038|-0.001|0.038   |m/s2  |phyphox|1791519187857|NULL     |NULL            |NULL            |2026-10-08 23:13:07.857|
+--------------+----------------+------+------+------+--------+------+-------+-------------+------

-------------------------------------------
Batch: 4
-------------------------------------------
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|tipoEvento    |bikeId          |accX  |accY  |accZ  |magnitud|unidad|origen |timestamp    |stationId|bicisDisponibles|docksDisponibles|ts                     |
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|-0.04 |-0.054|-0.027|0.072   |m/s2  |phyphox|1791519189929|NULL     |NULL            |NULL            |2026-10-08 23:13:09.929|
|bici.vibracion|bici-grimaldo-01|0.008 |0.085 |-0.053|0.1     |m/s2  |phyphox|1791519188907|NULL     |NULL            |NULL            |2026-10-08 23:13:08.907|
|bici.vibracion|bici-grimaldo-01|-0.194|0.052 |-0.051|0.207   |m/s2  |phyphox|1791519190946|NULL  

...

-------------------------------------------
Batch: 7
-------------------------------------------
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|tipoEvento    |bikeId          |accX  |accY  |accZ  |magnitud|unidad|origen |timestamp    |stationId|bicisDisponibles|docksDisponibles|ts                     |
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|-0.105|-0.107|-0.008|0.15    |m/s2  |phyphox|1791519198720|NULL     |NULL            |NULL            |2026-10-08 23:13:18.72 |
|bici.vibracion|bici-grimaldo-01|0.015 |-0.044|-0.038|0.06    |m/s2  |phyphox|1791519199749|NULL     |NULL            |NULL            |2026-10-08 23:13:19.749|
+--------------+----------------+------+------+------+--------+------+-------+-------------+------

-------------------------------------------
Batch: 8
-------------------------------------------
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|tipoEvento    |bikeId          |accX  |accY  |accZ  |magnitud|unidad|origen |timestamp    |stationId|bicisDisponibles|docksDisponibles|ts                     |
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|0.075 |-0.083|0.241 |0.265   |m/s2  |phyphox|1791519201856|NULL     |NULL            |NULL            |2026-10-08 23:13:21.856|
|bici.vibracion|bici-grimaldo-01|-0.001|-0.02 |0.031 |0.037   |m/s2  |phyphox|1791519200787|NULL     |NULL            |NULL            |2026-10-08 23:13:20.787|
|bici.vibracion|bici-grimaldo-01|-0.009|0.019 |-0.015|0.026   |m/s2  |phyphox|1791519202911|NULL  

-------------------------------------------
Batch: 9
-------------------------------------------
+--------------+----------------+------+-----+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|tipoEvento    |bikeId          |accX  |accY |accZ  |magnitud|unidad|origen |timestamp    |stationId|bicisDisponibles|docksDisponibles|ts                     |
+--------------+----------------+------+-----+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|-0.023|0.026|-0.042|0.054   |m/s2  |phyphox|1791519203964|NULL     |NULL            |NULL            |2026-10-08 23:13:23.964|
|bici.vibracion|bici-grimaldo-01|-0.034|0.002|0.028 |0.044   |m/s2  |phyphox|1791519205121|NULL     |NULL            |NULL            |2026-10-08 23:13:25.121|
|bici.vibracion|bici-grimaldo-01|0.007 |0.002|0.031 |0.032   |m/s2  |phyphox|1791519206174|NULL     |NU

-------------------------------------------
Batch: 10
-------------------------------------------
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|tipoEvento    |bikeId          |accX  |accY  |accZ  |magnitud|unidad|origen |timestamp    |stationId|bicisDisponibles|docksDisponibles|ts                     |
+--------------+----------------+------+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|0.011 |-0.01 |-0.01 |0.017   |m/s2  |phyphox|1791519207329|NULL     |NULL            |NULL            |2026-10-08 23:13:27.329|
|bici.vibracion|bici-grimaldo-01|0.045 |-0.032|-0.033|0.064   |m/s2  |phyphox|1791519208381|NULL     |NULL            |NULL            |2026-10-08 23:13:28.381|
|bici.vibracion|bici-grimaldo-01|-0.022|-0.018|0.01  |0.03    |m/s2  |phyphox|1791519209430|NULL 

-------------------------------------------
Batch: 11
-------------------------------------------
+--------------+----------------+-----+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|tipoEvento    |bikeId          |accX |accY  |accZ  |magnitud|unidad|origen |timestamp    |stationId|bicisDisponibles|docksDisponibles|ts                     |
+--------------+----------------+-----+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|0.003|0.012 |-0.001|0.013   |m/s2  |phyphox|1791519210825|NULL     |NULL            |NULL            |2026-10-08 23:13:30.825|
|bici.vibracion|bici-grimaldo-01|0.006|-0.018|-0.004|0.019   |m/s2  |phyphox|1791519211884|NULL     |NULL            |NULL            |2026-10-08 23:13:31.884|
|bici.vibracion|bici-grimaldo-01|0.012|-0.006|-0.01 |0.016   |m/s2  |phyphox|1791519210238|NULL     |N

In [9]:
n_salida = spark.sql(f"SELECT count(*) FROM dedup_mem WHERE bikeId = '{BIKE_ID}' AND timestamp = {t_dup}").first()[0]
n_kafka = lecturas_de_la_bici().filter(col("t") == t_dup).count()

spark.sql(f"SELECT * FROM dedup_mem WHERE timestamp = {t_dup}").show(truncate=False)
print("veces en Kafka (original + 2 reenvíos):", n_kafka)
print("veces en la salida deduplicada:      ", n_salida)
resumen["4. Duplicado"] = f"{n_kafka} copias en Kafka -> {n_salida} en la salida"

+--------------+----------------+-----+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|tipoEvento    |bikeId          |accX |accY  |accZ  |magnitud|unidad|origen |timestamp    |stationId|bicisDisponibles|docksDisponibles|ts                     |
+--------------+----------------+-----+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|0.046|-0.001|-0.067|0.082   |m/s2  |phyphox|1791519193113|NULL     |NULL            |NULL            |2026-10-08 23:13:13.113|
+--------------+----------------+-----+------+------+--------+------+-------+-------------+---------+----------------+----------------+-----------------------+

veces en Kafka (original + 2 reenvíos): 3
veces en la salida deduplicada:       1


## 5. Intervalos de disparo y contenido del checkpoint

### 5.1 Mismo stream, dos ritmos de disparo

La función `contar_microlotes` de la guía (3.12 en el notebook de práctica) sobre `vibracion`: 15 s de reloj con disparo cada 1 s y otros 15 s con disparo cada 6 s.

In [10]:
def contar_microlotes(intervalo, segundos_totales):
    consulta = (
        vibracion.writeStream
        .outputMode("append")
        .format("memory")
        .queryName(f"conteo_{intervalo.replace(' ', '_')}")
        .trigger(processingTime=intervalo)
        .start()
    )
    consulta.awaitTermination(segundos_totales)
    consulta.stop()
    return consulta.lastProgress["batchId"] if consulta.lastProgress else None

ultimo_lote_rapido = contar_microlotes("1 second", 15)
ultimo_lote_lento = contar_microlotes("6 seconds", 15)

filas_rapido = spark.sql("SELECT count(*) FROM conteo_1_second").first()[0]
filas_lento = spark.sql("SELECT count(*) FROM conteo_6_seconds").first()[0]
print("Con disparo cada 1 segundo, último batchId en 15s:", ultimo_lote_rapido, f"({filas_rapido} lecturas)")
print("Con disparo cada 6 segundos, último batchId en 15s:", ultimo_lote_lento, f"({filas_lento} lecturas)")
resumen["5. Disparo 1s vs 6s"] = f"batchId {ultimo_lote_rapido} vs {ultimo_lote_lento} en 15 s"

26/10/09 04:13:50 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 13, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@2b789f3f] is aborting.
26/10/09 04:13:50 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 13, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@2b789f3f] aborted.
26/10/09 04:13:50 ERROR Utils: Aborting task
org.apache.spark.TaskKilledException: Stage cancelled: [SPARK_JOB_CANCELLED] Job 124 cancelled Query conteo_1_second [id = 6a9395cb-2614-4720-bad9-e95860507013, runId = cfd5d45d-23a4-4fee-92ed-6c464a29ef82] was stopped SQLSTATE: XXKDA
	at org.apache.spark.TaskContextImpl.killTaskIfInterrupted(TaskContextImpl.scala:342)
	at org.apache.spark.InterruptibleIterator.hasNext(InterruptibleIterator.scala:36)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:610)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at scala.collec

Con disparo cada 1 segundo, último batchId en 15s: 12 (12 lecturas)
Con disparo cada 6 segundos, último batchId en 15s: 2 (9 lecturas)


Con disparo cada 1 s hay más micro-lotes de 1-2 lecturas cada uno: menor latencia, pero el *overhead* de coordinar se paga más veces. Con 6 s hay pocos lotes de ~6 lecturas: más throughput por lote, a costa de que cada lectura espere hasta 6 s antes de procesarse. Para una alerta de golpe o caída de la bici convendría el disparo corto.

### 5.2 Qué guardó el checkpoint de la sección 2.1

In [11]:
for carpeta in sorted(os.listdir(CHK_VENTANAS)):
    ruta = os.path.join(CHK_VENTANAS, carpeta)
    detalle = f"{len(os.listdir(ruta))} archivos" if os.path.isdir(ruta) else "archivo"
    print(f"{carpeta:15s} {detalle}")

archivos_offsets = sorted((f for f in os.listdir(f"{CHK_VENTANAS}/offsets") if not f.startswith(".")), key=int)
ultimo_offset = archivos_offsets[-1]
print(f"\n--- offsets/{ultimo_offset} ---")
with open(f"{CHK_VENTANAS}/offsets/{ultimo_offset}") as f:
    contenido = f.read()
print(contenido)

offsets_kafka = json.loads(contenido.strip().splitlines()[-1])[TOPIC]
print("particiones registradas:", sorted(offsets_kafka, key=int), "->", offsets_kafka)
print("operadores con estado en state/:", os.listdir(f"{CHK_VENTANAS}/state"))
resumen["5. Checkpoint"] = f"offsets/{ultimo_offset}: {len(offsets_kafka)} particiones {offsets_kafka}"

.metadata.crc   archivo
commits         16 archivos
metadata        archivo
offsets         16 archivos
sources         1 archivos
state           1 archivos

--- offsets/7 ---
v1
{"batchWatermarkMs":1791519024018,"batchTimestampMs":1791519050029,"conf":{"spark.sql.streaming.stateStore.providerClass":"org.apache.spark.sql.execution.streaming.state.HDFSBackedStateStoreProvider","spark.sql.streaming.stateStore.rocksdb.mergeOperatorVersion":"2","spark.sql.streaming.stateStore.rocksdb.formatVersion":"5","spark.sql.streaming.queryEvolution.enableSourceEvolution":"false","spark.sql.streaming.stateStore.encodingFormat":"unsaferow","spark.sql.streaming.statefulOperator.useStrictDistribution":"true","spark.sql.streaming.flatMapGroupsWithState.stateFormatVersion":"2","spark.sql.streaming.multipleWatermarkPolicy":"min","spark.sql.streaming.stateStore.rowChecksum.enabled":"false","spark.sql.streaming.aggregation.stateFormatVersion":"2","spark.sql.function.protobufExtensions.enabled":"false","spark

`offsets/` guarda, por micro-lote, hasta qué offset de cada partición se leyó (aquí las 3 de `citibike-eventos`) y el `batchWatermarkMs` vigente. `commits/` confirma qué micro-lotes terminaron de escribirse, y `state/` guarda las ventanas abiertas de la agregación. Si la consulta se reinicia con este mismo `checkpointLocation`, retoma desde esos offsets e ignora `startingOffsets`. Por eso cada sección de este notebook limpia su checkpoint antes de arrancar.

## 6. Persistencia en Parquet con checkpoint, en dos corridas

Se guardan los eventos **crudos** de ambos tipos (todas las columnas del esquema más `ts`). Un sink de archivo solo admite `append`. Al iniciar la sección se borran la carpeta y el checkpoint, para que la corrida 1 parta de cero en cada ejecución del notebook. La corrida 2 reutiliza la **misma ruta y el mismo checkpoint**.

In [12]:
ARTIFACTS = os.path.abspath("./artifacts/citibike_parquet")
CHK_PARQUET = os.path.abspath("./artifacts/chk_citibike_parquet")
limpiar(ARTIFACTS)
limpiar(CHK_PARQUET)

def corrida_parquet(segundos):
    consulta = (
        eventos.writeStream
        .outputMode("append")
        .format("parquet")
        .option("path", ARTIFACTS)
        .option("checkpointLocation", CHK_PARQUET)
        .trigger(processingTime="5 seconds")
        .start()
    )
    consulta.awaitTermination(segundos)
    consulta.stop()
    return spark.read.parquet(ARTIFACTS).count()

filas_corrida_1 = corrida_parquet(40)
print(f"corrida 1 = {filas_corrida_1} filas")

corrida 1 = 38 filas


In [13]:
filas_corrida_2 = corrida_parquet(40)
print(f"corrida 1 = {filas_corrida_1} filas")
print(f"corrida 2 = {filas_corrida_2} filas  ->", "acumula" if filas_corrida_2 > filas_corrida_1 else "NO acumuló")

guardado = spark.read.parquet(ARTIFACTS)
guardado.groupBy("tipoEvento").count().orderBy("tipoEvento").show(truncate=False)
guardado.orderBy(col("ts").desc()).select("tipoEvento", "bikeId", "magnitud", "stationId", "bicisDisponibles", "ts").show(6, truncate=False)
print("archivos .parquet en disco:", len([f for f in os.listdir(ARTIFACTS) if f.endswith(".parquet")]))
print("lotes confirmados en el checkpoint:", sorted(int(f) for f in os.listdir(f"{CHK_PARQUET}/commits") if f.isdigit()))
resumen["6. Parquet"] = f"corrida 1 = {filas_corrida_1}, corrida 2 = {filas_corrida_2} filas"

corrida 1 = 38 filas
corrida 2 = 81 filas  -> acumula


+---------------+-----+
|tipoEvento     |count|
+---------------+-----+
|bici.vibracion |69   |
|estacion.estado|12   |
+---------------+-----+



+--------------+----------------+--------+---------+----------------+-----------------------+
|tipoEvento    |bikeId          |magnitud|stationId|bicisDisponibles|ts                     |
+--------------+----------------+--------+---------+----------------+-----------------------+
|bici.vibracion|bici-grimaldo-01|0.024   |NULL     |NULL            |2026-10-08 23:15:19.417|
|bici.vibracion|bici-grimaldo-01|0.016   |NULL     |NULL            |2026-10-08 23:15:18.363|
|bici.vibracion|bici-grimaldo-01|0.017   |NULL     |NULL            |2026-10-08 23:15:17.212|
|bici.vibracion|bici-grimaldo-01|0.026   |NULL     |NULL            |2026-10-08 23:15:16.12 |
|bici.vibracion|bici-grimaldo-01|0.021   |NULL     |NULL            |2026-10-08 23:15:15.104|
|bici.vibracion|bici-grimaldo-01|0.015   |NULL     |NULL            |2026-10-08 23:15:14.048|
+--------------+----------------+--------+---------+----------------+-----------------------+
only showing top 6 rows
archivos .parquet en disco: 20
lotes

## Resumen de resultados

In [14]:
spark.createDataFrame(list(resumen.items()), "seccion string, resultado string").show(truncate=False)

+---------------------------------+--------------------------------------------------------+
|seccion                          |resultado                                               |
+---------------------------------+--------------------------------------------------------+
|0. Lectura batch                 |1137 mensajes en 3 particiones                          |
|1. Esquema                       |12 campos + ts; 2 DataFrames (vibracion, estaciones)    |
|2. Ventana 10s + watermark 20s   |8 micro-lotes en 40 s, outputMode update                |
|2. Deslizante estaciones 2min/30s|12 micro-lotes en 65 s, watermark 1 min                 |
|3. Dato tardío                   |ts 2026-10-08 22:52:39.106 (Lima): descartado (0 filas) |
|4. Duplicado                     |3 copias en Kafka -> 1 en la salida                     |
|5. Disparo 1s vs 6s              |batchId 12 vs 2 en 15 s                                 |
|5. Checkpoint                    |offsets/7: 3 particiones {'0': 223,